# 03 — Backtest Exploration (superseded)

**This notebook is exploratory scaffolding, not a result.** It predates
`scripts/run_backtest.py`, `metrics/performance.py` and `plotting/charts.py`, and it
reimplements the pipeline inline rather than importing it.

Two reasons its numbers must not be quoted as findings:

1. **It fits the static sizing hedge ratio on the full sample**
   (`estimate_hedge_ratio(lg[a], lg[b])`, cell below), so every out-of-sample trade is
   sized using information from the out-of-sample period. That is exactly the leak the
   study exists to avoid. `04_backtest_results.ipynb` fits it on the in-sample window only.
2. **It reports a single undivided sample**, with no IS/OOS split.

What it is genuinely useful for, and what the writeup cites it for, is the *diagnostic*
work: the COVID and 2024 spike attributions, and the window-sensitivity sweeps that show
how much of the headline number a free choice of lookback would have bought us.

→ **For the study's actual results, see `04_backtest_results.ipynb`.**
→ **For the narrative and conclusions, see `05_writeup.ipynb`.**

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from pairs_teardown.data.loaders import load_or_download
from pairs_teardown.data.clean import handle_missing, align_prices, to_log_prices
from pairs_teardown.signals.spread import build_rolling_spread, rolling_zscore, rolling_hedge_ratio
from pairs_teardown.signals.rules import target_positions
from pairs_teardown.backtest.costs import CostModel
from pairs_teardown.backtest.engine import run_backtest
from pairs_teardown.stats.cointegration import estimate_hedge_ratio, build_spread

ALL_TICKERS = ["WM", "RSG", "FOXA", "FOX", "SPY", "VOO", "KO", "PEP", "MA", "V", "XOM", "CVX"]
prices_raw = load_or_download(
    ALL_TICKERS, "2015-01-01", "2024-12-31", cache_dir=Path("../data/raw")
)

pair_defs = [("WM", "RSG"), ("FOXA", "FOX"), ("SPY", "VOO"), ("KO", "PEP"),("MA", "V"), ("XOM", "CVX")]
pairs = {
    f"{a}/{b}": align_prices(handle_missing(prices_raw[[a, b]]))
    for a, b in pair_defs
}

for name, df in pairs.items():
    print(f"{name}: {len(df)} days, {df.index[0].date()} → {df.index[-1].date()}")

In [ ]:
WINDOW, ENTRY, EXIT = 60, 2.0, 0.5
costs = CostModel(commission_bps=1.0, slippage_bps=5.0)   # 6 bps per side

results = {}
for a, b in pair_defs:
    name = f"{a}/{b}"
    raw = pairs[name]
    lg = to_log_prices(raw)

    # Signal: rolling hedge ratio, responsive to drift (unchanged)
    rolling_hr = rolling_hedge_ratio(lg[a], lg[b], WINDOW)
    z = rolling_zscore(build_rolling_spread(lg[a], lg[b], WINDOW), WINDOW)
    pos = target_positions(z, ENTRY, EXIT)

    # Sizing: STATIC hedge ratio, stable, no estimation-noise blowup
    static_hr_value = estimate_hedge_ratio(lg[a], lg[b])
    static_hr = pd.Series(static_hr_value, index=raw.index)

    results[name] = run_backtest(raw[a], raw[b], pos, static_hr, costs)

    print(f"{name}:")
    print(f"  rolling hedge ratio  mean={rolling_hr.mean():.4f}  std={rolling_hr.std():.4f}")
    print(f"  static hedge ratio={static_hr_value:.4f}")

In [ ]:
fig, axes = plt.subplots(6, 1, figsize=(12, 20))
for ax, (a, b) in zip(axes, pair_defs):
    name = f"{a}/{b}"
    res = results[name]
    idx = res.equity_curve.index
    ax.plot(idx, (1 + res.gross_returns).cumprod(), label="gross (no costs)", color="steelblue")
    ax.plot(idx, res.equity_curve, label="net (after costs)", color="firebrick")
    ax.axhline(1.0, color="black", lw=0.6)
    ax.set_title(f"{name}:  net total return = "
                 f"{(res.equity_curve.iloc[-1] - 1):.1%},  trades = {res.n_trades}")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### SPY/VOO 2020 spike investigation

In [ ]:
res = results["SPY/VOO"]

# Isolate the COVID window
covid_window = res.returns.loc["2020-02-01":"2020-04-30"]
full_sample = res.returns

print(f"Gross return, COVID window only:  {(1+res.gross_returns.loc['2020-02-01':'2020-04-30']).prod()-1:.2%}")
print(f"Gross return, full sample:         {(1+res.gross_returns).prod()-1:.2%}")
print(f"Gross return, full sample EXCLUDING COVID window:")
outside = res.gross_returns.drop(res.gross_returns.loc["2020-02-01":"2020-04-30"].index)
print(f"  {(1+outside).prod()-1:.2%}")

# Plot the position and z-score during that window to see what fired
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(res.held_positions.loc["2020-01-01":"2020-06-30"], drawstyle="steps-post")
ax.set_title("SPY/VOO held position, Jan–Jun 2020")
ax.set_yticks([-1, 0, 1])
plt.show()

### KO/PEP 2024 spike investigation

In [ ]:
res = results["KO/PEP"]

window_2024 = res.gross_returns.loc["2024-01-01":"2024-12-31"]
outside_2024 = res.gross_returns.drop(window_2024.index)

print(f"Gross return, 2024 only:              {(1+window_2024).prod()-1:.2%}")
print(f"Gross return, full sample:             {(1+res.gross_returns).prod()-1:.2%}")
print(f"Gross return, full sample EXCLUDING 2024:")
print(f"  {(1+outside_2024).prod()-1:.2%}")

# Position/z-score during the breakout, for visual confirmation
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(res.held_positions.loc["2023-06-01":"2024-12-31"], drawstyle="steps-post")
ax.set_title("KO/PEP held position, Jun 2023–Dec 2024")
ax.set_yticks([-1, 0, 1])
plt.show()

### Static hedge ratio (signal), trailing window sweep

In [ ]:
static_signal_results = {}
for a, b in pair_defs:
    name = f"{a}/{b}"
    raw = pairs[name]
    lg = to_log_prices(raw)

    static_hr_val = estimate_hedge_ratio(lg[a], lg[b])
    static_hr = pd.Series(static_hr_val, index=raw.index)

    static_spread = build_spread(lg[a], lg[b], static_hr_val)  # deterministic, no window

    for window in [40, 60, 90, 120]:
        z = rolling_zscore(static_spread, window)  # ONLY the z-score window varies now
        pos = target_positions(z, ENTRY, EXIT)
        res = run_backtest(raw[a], raw[b], pos, static_hr, costs)
        static_signal_results[(name, window)] = res.equity_curve.iloc[-1] - 1

for (name, w), ret in static_signal_results.items():
    print(f"{name:12s} window={w:3d}  net={ret:+.1%}")

### Rolling hedge ratio (signal), trailing window sweep

In [ ]:
grid = [40, 50, 60, 75, 90, 120]
sweep_results = {}

for a, b in pair_defs:
    name = f"{a}/{b}"
    raw = pairs[name]
    lg = to_log_prices(raw)
    static_hr_val = estimate_hedge_ratio(lg[a], lg[b])
    static_hr = pd.Series(static_hr_val, index=raw.index)

    for window in grid:
        z = rolling_zscore(build_rolling_spread(lg[a], lg[b], window), window)
        pos = target_positions(z, 2.0, 0.5)
        res = run_backtest(raw[a], raw[b], pos, static_hr, costs)
        sweep_results[(name, window)] = res.equity_curve.iloc[-1] - 1

sweep_table = pd.Series(sweep_results).unstack()
print(sweep_table.round(3))
print("\nSign changes across the grid, per pair:")
print(sweep_table.apply(lambda row: (row > 0).nunique() > 1, axis=1))

In [ ]:
diag_rows = []
for a, b in pair_defs:
    name = f"{a}/{b}"
    raw = pairs[name]
    lg = to_log_prices(raw)
    static_hr_val = estimate_hedge_ratio(lg[a], lg[b])
    static_hr = pd.Series(static_hr_val, index=raw.index)
    static_spread = build_spread(lg[a], lg[b], static_hr_val)

    for window in [40, 60, 90, 120]:
        z = rolling_zscore(static_spread, window)
        pos = target_positions(z, 2.0, 0.5)
        res = run_backtest(raw[a], raw[b], pos, static_hr, costs)
        gross_total = (1 + res.gross_returns).prod() - 1
        net_total = res.equity_curve.iloc[-1] - 1
        diag_rows.append({
            "pair": name, "window": window,
            "gross": gross_total, "net": net_total,
            "trades": res.n_trades, "cost_drag": gross_total - net_total,
        })

diag_df = pd.DataFrame(diag_rows)
print("Trade count by window:")
print(diag_df.pivot(index="pair", columns="window", values="trades"))
print("\nGROSS return by window (the key check):")
print(diag_df.pivot(index="pair", columns="window", values="gross").round(3))
print("\nCost drag (gross - net) by window:")
print(diag_df.pivot(index="pair", columns="window", values="cost_drag").round(3))